# Chapter 3 Companion Notebook: Financial Markets, Institutions, and Instruments

This notebook reproduces the quantitative examples from Chapter 3 of *AI in Finance* (`chapter3.tex`): the bid-ask spread as a measure of trading cost, walking the limit order book to see price impact from order size, the simple no-arbitrage argument connecting a risk-free bond to the price of a certain-payoff claim, short-selling margin calls, interest rate swap valuation, forward-contract cash-and-carry arbitrage, triangular currency arbitrage, and repo interest and futures variation margin.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. The bid-ask spread

The bid-ask spread is the simplest measure of the cost of trading immediately, rather than waiting for a better price.

In [1]:
bid = 100.20
ask = 100.35

spread = ask - bid
mid = (bid + ask) / 2
spread_bps = spread / mid * 10_000

print(f"Bid: ${bid:.2f}, Ask: ${ask:.2f}")
print(f"Spread: ${spread:.2f} ({spread_bps:.1f} bps of the mid price)")

Bid: $100.20, Ask: $100.35
Spread: $0.15 (15.0 bps of the mid price)


A round-trip trade (buy then immediately sell) at the quoted prices costs the full spread. Compare a liquid, tightly-quoted stock to an illiquid one:

In [2]:
import pandas as pd

quotes = pd.DataFrame({
    'Bid': [100.20, 24.90, 8.10],
    'Ask': [100.35, 25.30, 8.70],
}, index=['Liquid large-cap', 'Mid-cap', 'Illiquid small-cap'])

quotes['Spread'] = quotes['Ask'] - quotes['Bid']
quotes['Mid'] = (quotes['Bid'] + quotes['Ask']) / 2
quotes['Spread (bps)'] = quotes['Spread'] / quotes['Mid'] * 10_000
quotes.round(2)

,Bid,Ask,Spread,Mid,Spread (bps)
Liquid large-cap,100.2,100.35,0.15,100.28,14.96
Mid-cap,24.9,25.30,0.40,25.10,159.36
Illiquid small-cap,8.1,8.70,0.60,8.40,714.29


Notice how much larger the relative (basis-point) spread is for the illiquid small-cap name, even though its dollar spread looks modest -- this is exactly why comparing raw dollar spreads across differently priced stocks can be misleading, and why practitioners usually compare spreads in basis points.

## 2. Walking the limit order book

A large market order can consume several price levels, receiving a progressively worse average price than the best quote alone would suggest.

In [3]:
order_book_asks = pd.DataFrame({
    "Price": [50.10, 50.15, 50.20],
    "Size": [100, 200, 300],
}, index=[1, 2, 3])

best_bid, best_ask = 50.05, 50.10
quoted_spread = best_ask - best_bid
print(f"Best bid: ${best_bid:.2f}, Best ask: ${best_ask:.2f}, Quoted spread: ${quoted_spread:.2f}")

# A market buy order for 250 shares walks through two ask levels
order_size = 250
filled = 0
cost = 0.0
for price, size in zip(order_book_asks["Price"], order_book_asks["Size"]):
    take = min(size, order_size - filled)
    cost += take * price
    filled += take
    if filled >= order_size:
        break

avg_execution_price = cost / order_size
price_impact_bps = (avg_execution_price - best_ask) / best_ask * 10_000
print(f"Average execution price for a {order_size}-share buy: ${avg_execution_price:.2f}")
print(f"Price impact above best ask: {price_impact_bps:.1f} bps")

Best bid: $50.05, Best ask: $50.10, Quoted spread: $0.05
Average execution price for a 250-share buy: $50.13
Price impact above best ask: 6.0 bps


## 3. A simple no-arbitrage example

A one-period risk-free bond costs \$100 today and pays \$105 in one year.

In [4]:
bond_price = 100
bond_payoff = 105
risk_free_rate = (bond_payoff - bond_price) / bond_price
print(f"Implied risk-free rate: {risk_free_rate:.2%}")

Implied risk-free rate: 5.00%


Now suppose a separate claim pays exactly \$1 with certainty in one year (a zero-coupon bond with face value \$1). No-arbitrage says its price must be consistent with the same risk-free rate.

In [5]:
no_arbitrage_price = 1 / (1 + risk_free_rate)
print(f"No-arbitrage price of the $1 claim: ${no_arbitrage_price:.4f}")

# Suppose the claim is mispriced in the market at $0.95
market_price = 0.95
implied_return = (1 - market_price) / market_price
print(f"Market price: ${market_price:.2f}")
print(f"Implied return if bought at the market price: {implied_return:.2%}")
print(f"Risk-free rate available from the bond:        {risk_free_rate:.2%}")

if implied_return > risk_free_rate:
    print("Arbitrage: buy the claim, fund it by borrowing at the risk-free rate, "
          "and pocket the difference risk-free.")

No-arbitrage price of the $1 claim: $0.9524
Market price: $0.95
Implied return if bought at the market price: 5.26%
Risk-free rate available from the bond:        5.00%
Arbitrage: buy the claim, fund it by borrowing at the risk-free rate, and pocket the difference risk-free.


This mispricing cannot persist: as more investors exploit it, buying pressure on the claim and selling pressure on the bond (or direct borrowing) push the claim's price up toward the no-arbitrage value of $1/(1+r_f)$, closing the gap.

## 4. Short selling and margin calls (Section 3.11)

A trader shorts 100 shares at $50, with a 150% initial margin requirement and a 25% maintenance margin. Find the price at which a margin call is triggered.

In [6]:
shares = 100
P0 = 50
initial_margin_pct = 0.50   # additional margin beyond the 100% proceeds, i.e. 150% total
maintenance_pct = 0.25

proceeds = shares * P0
initial_deposit = initial_margin_pct * proceeds
total_equity0 = proceeds + initial_deposit

# Equity(P) = total_equity0 - shares*P; margin call when Equity(P)/(shares*P) = maintenance_pct
P_call = total_equity0 / (shares * (1 + maintenance_pct))
equity_at_call = total_equity0 - shares * P_call

print(f"Proceeds: ${proceeds:,.2f}, initial deposit: ${initial_deposit:,.2f}, total equity: ${total_equity0:,.2f}")
print(f"Margin call price: ${P_call:.2f}")
print(f"Equity at call: ${equity_at_call:,.2f} ({equity_at_call/(shares*P_call):.2%} of liability)")

Proceeds: $5,000.00, initial deposit: $2,500.00, total equity: $7,500.00
Margin call price: $60.00
Equity at call: $1,500.00 (25.00% of liability)


## 4b. Valuing an existing interest rate swap (Section 3.5)

A 2-year, $10M-notional swap with a 5% fixed rate, valued today (right after a reset) when 1-year and 2-year discount rates are 4% and 4.5%.

In [7]:
notional_swap = 10_000_000
fixed_rate = 0.05
r1, r2 = 0.04, 0.045

fixed_coupon = notional_swap * fixed_rate
pv_fixed = fixed_coupon / (1 + r1) + (fixed_coupon + notional_swap) / (1 + r2) ** 2
pv_floating = notional_swap

value_to_fixed_payer = pv_floating - pv_fixed
print(f"PV(fixed leg): ${pv_fixed:,.2f}")
print(f"PV(floating leg): ${pv_floating:,.2f}")
print(f"Swap value to fixed-rate payer: ${value_to_fixed_payer:,.2f}")

# Exercise: rates rise to 5.5% / 6% instead of falling
r1_ex, r2_ex = 0.055, 0.06
pv_fixed_ex = fixed_coupon / (1 + r1_ex) + (fixed_coupon + notional_swap) / (1 + r2_ex) ** 2
value_to_fixed_payer_ex = pv_floating - pv_fixed_ex
print(f"\nExercise (rates rise) -- PV(fixed leg): ${pv_fixed_ex:,.2f}, "
      f"value to fixed-rate payer: ${value_to_fixed_payer_ex:,.2f}")

PV(fixed leg): $10,095,933.72
PV(floating leg): $10,000,000.00
Swap value to fixed-rate payer: $-95,933.72

Exercise (rates rise) -- PV(fixed leg): $9,818,896.27, value to fixed-rate payer: $181,103.73


## 5. Forward contract no-arbitrage (Section 3.15)

A non-dividend-paying stock trades at $100, and the risk-free rate is 5%. Compute the no-arbitrage one-year forward price, and the riskless cash-and-carry profit if the forward instead trades at $110.

In [8]:
S0 = 100
r = 0.05
F_fair = S0 * (1 + r)

F_market = 110
loan_repayment = S0 * (1 + r)
arbitrage_profit = F_market - loan_repayment

print(f"No-arbitrage forward price: ${F_fair:.2f}")
print(f"Market forward price: ${F_market:.2f}")
print(f"Riskless cash-and-carry profit: ${arbitrage_profit:.2f}")

No-arbitrage forward price: $105.00
Market forward price: $110.00
Riskless cash-and-carry profit: $5.00


### 5b. Triangular arbitrage in currency markets (Section 3.15)

EUR/USD=1.10, GBP/USD=1.25, and a quoted EUR/GBP=0.87 (vs. the 0.88 implied by no-arbitrage). Trace the profitable GBP->EUR->USD->GBP cycle.

In [9]:
eur_usd = 1.10
gbp_usd = 1.25
eur_gbp_quoted = 0.87

eur_gbp_implied = eur_usd / gbp_usd
print(f"Implied EUR/GBP: {eur_gbp_implied:.4f}, quoted: {eur_gbp_quoted}")

start_gbp = 1_000_000
eur_amount = start_gbp / eur_gbp_quoted
usd_amount = eur_amount * eur_usd
end_gbp = usd_amount / gbp_usd
profit = end_gbp - start_gbp
print(f"GBP {start_gbp:,.2f} -> EUR {eur_amount:,.2f} -> USD {usd_amount:,.2f} -> GBP {end_gbp:,.2f}")
print(f"Profit: GBP {profit:,.2f} ({profit/start_gbp:.2%})")

# Exercise: EUR/GBP quoted at 0.89 instead of 0.87
eur_gbp_ex = 0.89
eur_amount_ex = start_gbp / eur_gbp_ex
usd_amount_ex = eur_amount_ex * eur_usd
end_gbp_ex = usd_amount_ex / gbp_usd
profit_ex = end_gbp_ex - start_gbp
print(f"\nExercise (EUR/GBP=0.89) -- same cycle end value: GBP {end_gbp_ex:,.2f}, "
      f"profit: GBP {profit_ex:,.2f} (still profitable? {profit_ex > 0})")

Implied EUR/GBP: 0.8800, quoted: 0.87
GBP 1,000,000.00 -> EUR 1,149,425.29 -> USD 1,264,367.82 -> GBP 1,011,494.25
Profit: GBP 11,494.25 (1.15%)

Exercise (EUR/GBP=0.89) -- same cycle end value: GBP 988,764.04, profit: GBP -11,235.96 (still profitable? False)


## 6. Repo interest and futures variation margin (Section 3.13, Section 3.17)

Two related money-market/clearing calculations: overnight repo interest on a $10 million loan at a 5% annualized rate (360-day convention), and the variation margin call on a $50,000-notional futures position after a 2% adverse price move.

In [10]:
# Overnight repo interest
principal = 10_000_000
repo_rate = 0.05
days = 1
repo_interest = principal * repo_rate * days / 360
repurchase_price = principal + repo_interest
print(f"Repo interest for one night: ${repo_interest:,.2f}")
print(f"Repurchase price: ${repurchase_price:,.2f}")

# Futures variation margin
notional = 50_000
initial_margin = 5_000
price_change_pct = -0.02
mtm_loss = notional * abs(price_change_pct)
new_balance = initial_margin - mtm_loss
variation_margin_call = initial_margin - new_balance
print(f"\nMark-to-market loss: ${mtm_loss:,.2f}")
print(f"Margin balance after move: ${new_balance:,.2f}")
print(f"Variation margin call: ${variation_margin_call:,.2f}")

Repo interest for one night: $1,388.89
Repurchase price: $10,001,388.89

Mark-to-market loss: $1,000.00
Margin balance after move: $4,000.00
Variation margin call: $1,000.00


## Exercises (match Chapter 3, Suggested Exercises)

1. Compute the bid-ask spread in both dollar and basis-point terms for a stock quoted at a bid of \$49.85 and an ask of \$50.05.
2. A one-period risk-free bond costs \$1,000 and pays \$1,040 in one year. A certain \$1 claim trades at \$0.965. Is there an arbitrage opportunity? If so, describe the trade that exploits it.

In [11]:
# Exercise 1
bid_ex, ask_ex = 49.85, 50.05
spread_ex = ask_ex - bid_ex
mid_ex = (bid_ex + ask_ex) / 2
print(f"Spread: ${spread_ex:.2f} ({spread_ex/mid_ex*10_000:.1f} bps)")

# Exercise 2
rf_ex = (1040 - 1000) / 1000
claim_price_ex = 0.965
no_arb_price_ex = 1 / (1 + rf_ex)
print(f"Risk-free rate: {rf_ex:.2%}")
print(f"No-arbitrage price of $1 claim: ${no_arb_price_ex:.4f}")
print(f"Market price: ${claim_price_ex:.3f} -> "
      f"{'underpriced, buy the claim' if claim_price_ex < no_arb_price_ex else 'overpriced, sell/short the claim'}")

Spread: $0.20 (40.0 bps)
Risk-free rate: 4.00%
No-arbitrage price of $1 claim: $0.9615
Market price: $0.965 -> overpriced, sell/short the claim
